In [7]:
# 加载环境变量
import os

DEEPSEEK_API = os.getenv("DEEPSEEK_API")
TAVILY_KEY = os.getenv("TAVILY_KEY")

In [8]:
# 创建模型
from langchain.chat_models import init_chat_model

research_model = init_chat_model(
    model="openai:deepseek-v4-pro", base_url="https://api.deepseek.com/v1", api_key=DEEPSEEK_API, temperature=0.7
)

main_model = init_chat_model(
    model="openai:deepseek-v4-flash", base_url="https://api.deepseek.com/v1", api_key=DEEPSEEK_API, temperature=0.7
)

In [9]:
# 创建工具
from typing import Literal

from tavily import TavilyClient

tavily_clent = TavilyClient(api_key=TAVILY_KEY)


def internet_search(
    query: str,
    max_results: int = 5,
    topic: Literal["general", "news", "finance"] = "general",
    include_raw_content: bool = False,
):
    """互联网搜索工具，当需要通过互联网搜索信息是使用

    Args:
        query (str): 查询内容
        max_results (int, optional): 最大结果数量. Defaults to 5.
        topic (Literal[&quot;general&quot;, &quot;news&quot;, &quot;finace&quot;], optional): 主题类型. Defaults to "general".
        include_raw_content (bool, optional): 是否包含原始内容. Defaults to False.

    Returns:
        _type_: _description_
    """
    return tavily_clent.search(
        query=query,
        max_results=max_results,
        include_raw_content=include_raw_content,
        topic=topic,
    )

In [10]:
# 定义子智能体

research_subagent = {
    "name": "research-subagent",
    "description": (
        "用于处理需要多轮检索和综合分析的深入研究任务，",
        "如 '系统性梳理某个技术方向', '对比多个方案优劣'等",
    ),
    "system_prompt": """
    你是一个严谨的研究员， 负责对某个主题进行深入调研并输出结构化结论。
    
    你的工作流程：
    1. 将问题拆分成若干可以检索的子问题；
    2. 调用 internet_search 工具多次检索信息；
    3. 对信息进行筛选， 归纳和对比分析；
    4. 最终输出: 简短结论 + 关键发现要点 + 可执行建议
    
    输出要求：
    - 使用中文回答；
    - 线给出 3～5 条要点总结， 在给出一个 2～3 端的综合说明；
    - 不要直接输出原始结果内容
    """,
    "tools": [internet_search],
    "model": research_model,
}

subagents = [research_subagent]

In [11]:
# 定义主智能体
from deepagents import create_deep_agent

main_agent = create_deep_agent(
    model=main_model,
    system_prompt="""
    你是一个任务协调者， 负责理解用户需求，拆分任务，
    并在需要是将'深入研究类任务'交给 research_subagent 子智能体处理，
    
    当你觉得需要多次检索、信息较多时，
    请使用 task 工具调用 research_subagent， 并等待返回结果总结。
    最后， 你负责把结果用 Markdown 语言表达出来
    """,
    tools=[],
    subagents=subagents,
)

In [12]:
# 调用智能体
result = main_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": """
            请帮我系统性的调研一下无人机反制领域的前景。
            需要结合国家对相关行业的政策和法律变化，各厂家的技术和商业行为，区分军用/民用，政策/法律风险等。
            """,
        }
    ]
})

In [14]:
# 输入结果
from rich.console import Console
from rich.markdown import Markdown

console = Console()

console.print(Markdown(result["messages"][-1].content))

以下是系统性调研报告的核心成果。                                                                                   

-------------------------------------------------------------------------------------------------------------------

                                     无人机反制（C-UAS）领域前景系统性分析报告                                     

-------------------------------------------------------------------------------------------------------------------

一、核心结论                                                                                                       

反无人机领域正处于 从碎片化向体系化、从"应对威胁"向"塑造规则"转变 的关键阶段。2025-2030年全球市场将延续            
25%以上的高速增长，到2030年有望成为 200-300亿美元量级 的独立市场。                                                 

-------------------------------------------------------------------------------------------------------------------

二、政策与法律环境                                                                                                 

中国                                                                                                               

                                                                                                                   
 关键事件                                          内容                                                            
 ───────────────────────────────────────────────────────────────────────────────────────────────────────────────── 
 《无人驾驶航空器飞行管理暂行条例》（2024.1施行）  首部无人机管理专门行政法规，第52条明确禁止非法拥有/使用反制设 … 
 2025年中央空管委                                  将反无人机技术纳入全国低空安全监管体系，要求电磁干扰精度覆盖2.… 
 2025年商务部                                      对12家美国无人机企业实施出口管制                                
 2026年新《治安管理处罚法》                        首次将无人机违规飞行纳入妨害公共安全行为，情节较重可拘留5-10日  
                                                                                                                   

美国                                                                                                               

 • 《2025财年国防授权法》第1090条：指示国防部长制定反无人机系统战略                                                
 • 白宫"恢复美国空域主权"行政令（2025.6）：成立联邦任务组，推动反无人机立法                                        
 • 核心矛盾：联邦法律（《通信法》）严格限制民用场景使用主动反制手段，目前仅DHS和DOJ获试点授权                      

欧盟                                                                                                               

 • 2026年初发布《无人机及反无人机安全行动计划》，计划2030年前建立欧盟层面反无人机监管框架                          
 • 要求成员国2028年前在关键设施部署分层空域保护                                                                    

法律风险（核心制约因素）                                                                                           

                                                               
 风险类型          关键问题                                    
 ───────────────────────────────────────────────────────────── 
 无线电干扰合规    干扰2.4/5.8GHz频段可能违法，美国FCC严格限制 
 隐私与数据安全    反制系统会采集操控者位置、飞行轨迹等信息    
 武器出口管制      ITAR/中国出口管制对定向能等高端系统限制严格 
 误击/误伤责任     无人机坠落造成地面伤亡、干扰民航通信        
 与合法无人机冲突  低空经济（物流/eVTOL）反被反制系统误伤      
                                                               

-------------------------------------------------------------------------------------------------------------------

三、技术路线全景                                                                                                   

探测技术                                                                                                           

                                                                               
 技术        成熟度  优势                    局限                              
 ───────────────────────────────────────────────────────────────────────────── 
 雷达探测    ★★★★★   全天候、远距离、多目标  对微型/悬停目标弱，主动辐射易暴露 
 RF频谱监测  ★★★★    隐蔽性好，可定位操控者  对自主飞行无人机无效              
 光电/红外   ★★★★    可视化确认，精度高      受天气/光照影响大                 
 声学探测    ★★★     成本低、全被动          探测距离<300m，城市环境干扰大     
                                                                               

反制/拦截技术                                                                                            